# Notebook 01: Experiment with managed MLflow and scale with SageMaker AI jobs

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> Tested with <strong>SageMaker Distribution Image 4.3.2</strong> and <strong>SageMaker Python SDK 3.20.0</strong>.
</div>

In this notebook, you take an idea to a scaled, tracked, registered, and approved model in one pass, building on the environment from notebook 00. Serving that model is notebook 04.

|||
|---|---|
|00. Environment setup ||
|**01. Experiment in a notebook, then scale with SageMaker AI jobs** |**◄ YOU ARE HERE**|
|02. Automate model governance with EventBridge and Model Cards ||
|03. Operationalize with an ML pipeline and model registry ||
|04. Deploy to an endpoint and monitor ||
|05. Templatize staged deployment as a Project ||

## What you do here

1. **Part 1** — explore the bank-marketing dataset, engineer features, and train XGBoost locally in plain Python, tracking every run in [SageMaker managed MLflow](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html). No `sagemaker` job APIs and no registration: the prototype stays an experiment.
2. **Part 2** — run the same logic on SageMaker AI managed infrastructure, which serves processing, training, and inference through [Docker containers](https://docs.aws.amazon.com/sagemaker/latest/dg/docker-containers.html) that honor a fixed contract, using the custom XGBoost image (BYOC) from notebook 00.
3. Run preprocessing once as a job; its S3 datasets are the canonical inputs for notebooks 03 and 04.
4. Explain the model with SHAP and standardized fairness metrics as a managed job.
5. Register one deployable Model Package via inference-spec logging and `mlflow.register_model()`.
6. Stage the version as the data scientist, then promote and approve it as the governance officer, with IAM denying the data scientist that step.

**Outcome:** one registered Model Package version, approved for production, which notebook 04 deploys from the registry.

<div style="border: 4px solid #d32f2f; border-radius: 10px; padding: 20px; background-color: #ffebee; margin: 15px 0;">

### ⛔ IMPORTANT: Python 3.12 Required

All notebooks in this workshop **must** run on **Python 3.12**. The cell below will fail with an `AssertionError` if you are using a different Python version.

If you see an error, follow these steps:

1. Open a **Terminal** in JupyterLab (**File → New → Terminal**)
2. Run the setup script:
   ```bash
   cd ~/aim-408-end-to-end-modelops-on-sagemaker-ai
   ./setup-py312.sh
   ```
3. **Refresh your browser** to pick up the new kernel
4. In this notebook, go to **Kernel → Change Kernel → Python 3.12**
5. **Restart the kernel** and re-run from the top

</div>

In [ ]:
import sys

assert sys.version_info[:2] == (3, 12), f"Expected Python 3.12, got {sys.version_info.major}.{sys.version_info.minor}. Please make sure you run on Python 3.12"

### Check the installed SageMaker version

In [ ]:
import sagemaker
import sagemaker.core
import sagemaker.train
import sagemaker.serve
import sagemaker.mlops
from importlib.metadata import version

print(f"sagemaker: {version('sagemaker')}")
print(f"sagemaker.core: {version('sagemaker.core')}")
print(f"sagemaker.train: {version('sagemaker.train')}")
print(f"sagemaker.server: {version('sagemaker.serve')}")
print(f"sagemaker.mlops: {version('sagemaker.mlops')}")

### Import packages
One import cell for the whole notebook. The two `Mode` enums (training and serving) are aliased because the SDK gives them the same class name.

In [ ]:
import os
import time
import json
import tempfile
import warnings
from io import StringIO
from importlib.metadata import version

import boto3
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
import mlflow
import sagemaker_mlflow
from mlflow import MlflowClient
from IPython.display import HTML, Javascript
from sklearn import metrics
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score

from sagemaker.core.resources import MlflowApp, ModelPackage
from sagemaker.core.s3 import S3Downloader
from sagemaker.core.s3.client import s3_path_join
from sagemaker.core.common_utils import name_from_base
from sagemaker.core.utils.utils import configure_logging
from sagemaker.core.helper.session_helper import Session
from sagemaker.core.shapes import (
    ProcessingInput,
    ProcessingS3Input,
    ProcessingOutput,
    ProcessingS3Output,
    OutputDataConfig,
)
from sagemaker.core import FrameworkProcessor
from sagemaker.core.remote_function.client import remote
from sagemaker.core.remote_function.custom_file_filter import CustomFileFilter
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import InputData, SourceCode, Compute
from sagemaker.train.model_trainer import Mode as TrainingMode

# The explainability step is shared with notebook 03, where the same function
# is wrapped with @step instead of @remote.
from pipeline_steps.explain import explain
# Two-persona approval gate (data scientist vs governance officer), shared with notebook 03.
from governance.approval_gate import (
    DATA_SCIENTIST,
    GOVERNANCE_OFFICER,
    approve_for_deployment,
    assume_persona,
    deny_statements,
    describe_guardrail,
    freeze_model_package_group,
    load_persona_policy,
    model_package_group_tags,
    print_lifecycle,
    set_lifecycle,
    set_version_metadata,
    simulate_lifecycle_decision,
    tag_model_package_group,
    try_update_model_package,
    wait_for_lifecycle,
)

warnings.filterwarnings("ignore")
configure_logging("INFO")

Load the variables stored by previous notebooks.

In [ ]:
%store -r 

%store

try:
    initialized
except NameError:
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN 00-start-here notebook   ")
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")

Set and store the name of the dataset's target column.

In [ ]:
target_col = "y"
%store target_col

## Set up experiment tracking with managed MLflow
[SageMaker managed MLflow](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html) runs MLflow as a serverless app. [MLflow tracking](https://mlflow.org/docs/latest/tracking.html) records the inputs, parameters, configurations, and models of each iteration as `experiments` and `runs` (see [MLflow concepts](https://mlflow.org/docs/latest/tracking.html#concepts)).

Notebook 00 created (or reused) the MLflow app and stored its ARN in `mlflow_arn`. The cell below waits until the app is ready, which can take 2-3 minutes on a first run.

In [ ]:
mlflow_app = MlflowApp.get(arn=mlflow_arn)

max_wait_time = 300  # 5 minutes
start_time = time.time()

while mlflow_app.status in ["Creating", "Updating"]:
    if time.time() - start_time > max_wait_time:
        raise TimeoutError(f"MLflow app '{mlflow_name}' still {mlflow_app.status} after 5 minutes")
    print(f"MLflow app '{mlflow_name}' is {mlflow_app.status}... waiting 30 seconds")
    time.sleep(30)
    mlflow_app.refresh()

if mlflow_app.status not in ["Created", "Updated"]:
    raise RuntimeError(f"MLflow app '{mlflow_name}' is not usable (status: {mlflow_app.status})")

print(f"MLflow app '{mlflow_name}' is ready (status: {mlflow_app.status})")

Point MLflow at the app and set the experiment and registered-model names.

These are **shared names**: one experiment and one registered model for the whole workshop, suffixed with a short date stamp (`-DDMMYY`). They are created here on first use and persisted with `%store`, so every later notebook restores and reuses them. `set_experiment()` returns the existing experiment, and each `register_model()` call adds a new **version** rather than a new model.

In [ ]:
mlflow.set_tracking_uri(mlflow_arn)
print(f"Using MLflow server: {mlflow_arn}")

date_suffix = time.strftime("%d%m%y", time.gmtime())
experiment_name = f"from-idea-to-prod-experiment-{date_suffix}"
registered_model_name = f"from-idea-to-prod-model-{date_suffix}"

%store experiment_name
%store registered_model_name

experiment = mlflow.set_experiment(experiment_name=experiment_name)

print(f"Experiment: {experiment_name}")
print(f"Registered model: {registered_model_name}")

---
# Part 1 — Prototype in the notebook

## Load data
The dataset is already available locally from notebook 00.

In [ ]:
file_name = "bank-additional-full.csv"
input_path = "./data/bank-additional" 
output_path = "./data"

## EDA
Explore the dataset before modeling.

In [ ]:
df_data = pd.read_csv(os.path.join(input_path, file_name), sep=";")

pd.set_option("display.max_columns", 500)  # View all of the columns
df_data  # show first 5 and last 5 rows of the dataframe

Check the column metadata for null counts and data types.

In [ ]:
df_data.info()

Column statistics are a quick sanity check: age ranges 17-98 with a mean of 40, all reasonable.

In [ ]:
df_data.describe()

The target distribution is unbalanced: far more "no" than "yes" responses to the marketing calls.

In [ ]:
df_data[target_col].value_counts().plot.bar()
plt.show()

Check for missing values. This dataset is clean.

In [ ]:
df_data.isna().sum()

Distributions for the categorical columns: most customers are highly educated and married, and some have an unknown job.

In [ ]:
cat_columns = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

fig, axs = plt.subplots(3, 3, sharex=False, sharey=False, figsize=(20, 15))

counter = 0
for cat_column in cat_columns:
    value_counts = df_data[cat_column].value_counts()
    
    trace_x = counter // 3
    trace_y = counter % 3
    x_pos = np.arange(0, len(value_counts))
    
    axs[trace_x, trace_y].bar(x_pos, value_counts.values, tick_label = value_counts.index)
    
    axs[trace_x, trace_y].set_title(cat_column)
    
    for tick in axs[trace_x, trace_y].get_xticklabels():
        tick.set_rotation(90)
    
    counter += 1

plt.show()

Distributions for the numerical columns: call `duration` is almost always short, and most customers were not contacted in a previous campaign (`pdays`).

In [ ]:
num_columns = ['duration', 'campaign', 'pdays', 'previous']

fig, axs = plt.subplots(2, 2, sharex=False, sharey=False, figsize=(20, 15))

counter = 0
for num_column in num_columns:
    
    trace_x = counter // 2
    trace_y = counter % 2
    
    axs[trace_x, trace_y].hist(df_data[num_column])
    
    axs[trace_x, trace_y].set_title(num_column)
    
    counter += 1

plt.show()

Compare a column against the target, here marital status.

In [ ]:
j_df = pd.DataFrame()

j_df['yes'] = df_data[df_data[target_col] == 'yes']['marital'].value_counts()
j_df['no'] = df_data[df_data[target_col] == 'no']['marital'].value_counts()

j_df.plot.bar(title = 'Marital status and deposit')

## Feature engineering in the notebook
Prototype the transformations here; in Part 2 the same logic moves into `processing/preprocessing.py` and runs as a processing job.

The processing:
1. Adds `no_previous_contact`: `1` when `pdays` is `999`, else `0`
1. Adds a column for whether the customer is working, from the `job` column
1. Removes the economic features, which would need high-precision forecasts to be usable at inference time
1. Removes `duration`, which is not known before a call
1. One-hot encodes categorical variables
1. Moves the target column `y` to the front

In [ ]:
# Indicator variable to capture when pdays takes a value of 999
df_data["no_previous_contact"] = np.where(df_data["pdays"] == 999, 1, 0)

# Indicator for individuals not actively employed
df_data["not_working"] = np.where(
    np.isin(df_data["job"], ["student", "retired", "unemployed"]), 1, 0
)

# remove unnecessary data
df_model_data = df_data.drop(
    ["duration", "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"],
    axis=1,
)


bins = [18, 30, 40, 50, 60, 70, 90]
labels = ['18-29', '30-39', '40-49', '50-59', '60-69', '70-plus']

df_model_data['age_range'] = pd.cut(df_model_data.age, bins, labels=labels, include_lowest=True)
df_model_data = pd.concat([df_model_data, pd.get_dummies(df_model_data['age_range'], prefix='age', dtype=int)], axis=1)
df_model_data.drop('age', axis=1, inplace=True)
df_model_data.drop('age_range', axis=1, inplace=True)

scaled_features = ['pdays', 'previous', 'campaign']
df_model_data[scaled_features] = MinMaxScaler().fit_transform(df_model_data[scaled_features])

df_model_data = pd.get_dummies(df_model_data, dtype=int)  # Convert categorical variables to sets of indicators

# Replace "y_no" and "y_yes" with a single label column, and bring it to the front:
df_model_data = pd.concat(
    [
        df_model_data["y_yes"].rename(target_col),
        df_model_data.drop(["y_no", "y_yes"], axis=1),
    ],
    axis=1,
)

After preprocessing, re-inspect the dataset and column statistics as a sanity check.

In [ ]:
df_model_data

In [ ]:
df_model_data.describe()

## Split data

[SageMaker's XGBoost algorithm](https://docs.aws.amazon.com/sagemaker/latest/dg/xgboost.html#InputOutput-XGBoost), used in Part 2, expects libSVM or CSV with the target in the first column and no header row. The cells below shuffle and split the data into train, validation, and test sets, then save each as a correctly formatted CSV.

In [ ]:
df_shuffled = df_model_data.sample(frac=1, random_state=1729)
# Split data into 70% training, 20% validation, 10% test
train_idx = int(0.7 * len(df_shuffled))
val_idx = int(0.9 * len(df_shuffled))

train_data = df_shuffled.iloc[:train_idx]
validation_data = df_shuffled.iloc[train_idx:val_idx]
test_data = df_shuffled.iloc[val_idx:]

print(f"Data split > train:{train_data.shape} | validation:{validation_data.shape} | test:{test_data.shape}")

In [ ]:
train_data.to_csv(os.path.join(output_path, "train.csv"), index=False, header=False)
validation_data.to_csv(os.path.join(output_path, "validation.csv"), index=False, header=False)
test_data.to_csv(os.path.join(output_path, "test.csv"), index=False, header=False)

### Track the feature engineering in an MLflow run
Log data processing as well as training, so both live in the same experiment. Create a run with [`mlflow.start_run()`](https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.start_run), then record into it with [`log_params()`](https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.log_params) and [`log_artifact()`](https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.log_artifact).

Here you log the split ratios and the **raw input dataset**. The train/validation/test splits are logged later by the Part 2 processing job, so the same CSVs are not uploaded twice.

In [ ]:
with mlflow.start_run(
    run_name=name_from_base("feature-engineering", short=True),
    description="feature engineering prototyped in the notebook (Part 1)"
) as run:
    mlflow.set_tags({"mlflow.user": user_profile_name})
    mlflow.log_params(
        {
            "train": 0.7,
            "validate": 0.2,
            "test": 0.1
        }
    )
    # Log the raw input dataset. The train/validation/test splits produced from it
    # are logged by the SageMaker processing job in Part 2 (same data, logged once).
    mlflow.log_artifact(local_path=os.path.join(input_path, file_name))

## Model training and validation

Part 1 uses the [open-source XGBoost](https://xgboost.ai/about) to train inside the notebook. First, split the features from the label in the training dataset.

In [ ]:
train_features = train_data.drop(target_col, axis=1)
train_label = pd.DataFrame(train_data[target_col])

Create the data structure XGBoost accepts as input.

In [ ]:
dtrain = xgb.DMatrix(train_features, label=train_label)

Set your hyperparameters and other variables.

In [ ]:
hyperparams = {
                "max_depth": 5,
                "eta": 0.5,
                "alpha": 2.5,
                "objective": "binary:logistic",
                "subsample" : 0.8,
                "colsample_bytree" : 0.8,
                "min_child_weight" : 3
              }

num_boost_round = 150
nfold = 3
early_stopping_rounds = 10

Run cross-validation on `nfold` folds to see how well the model generalizes with the current hyperparameters.

In [ ]:
cv_results = xgb.cv(
    params=hyperparams,
    dtrain=dtrain,
    num_boost_round=num_boost_round,
    nfold=nfold,
    early_stopping_rounds=early_stopping_rounds,
    metrics=["auc"],
    seed=10,
)

Fetch and print the metrics to understand how well the model performs.

In [ ]:
metrics_data = {
    "binary_classification_metrics": {
        "validation:auc": {
            "value": cv_results.iloc[-1]["test-auc-mean"],
            "standard_deviation": cv_results.iloc[-1]["test-auc-std"]
        },
        "train:auc": {
            "value": cv_results.iloc[-1]["train-auc-mean"],
            "standard_deviation": cv_results.iloc[-1]["train-auc-std"]
        },
    }
}

print(f"Cross-validated train-auc:{cv_results.iloc[-1]['train-auc-mean']:.2f}")
print(f"Cross-validated validation-auc:{cv_results.iloc[-1]['test-auc-mean']:.2f}")
display(metrics_data)

cv_results

Now retrain on the full training dataset rather than across folds.

### Train a model
Use [MLflow model flavors](https://mlflow.org/docs/latest/python_api/index.html#python-api) and [logging functions](https://mlflow.org/docs/latest/tracking/tracking-api.html#tracking-logging-functions) to log parameters, the model, metrics, and metadata to your experiment runs.

The test dataset drives early stopping, so first split its features and label and build the data structure the algorithm expects.

In [ ]:
test_features = test_data.drop(target_col, axis=1)
test_label = pd.DataFrame(test_data[target_col])
dtest = xgb.DMatrix(test_features, label=test_label)

The training loop below trains the model at several `max_depth` values and logs each run to MLflow for comparison.

In [ ]:
max_metric = 0.0
best_model_run_id = 0

with mlflow.start_run(
    run_name=name_from_base("training-", short=True),
    description=f"Fit estimator with different max_depth"
) as parent_run:
    mlflow.set_tags({'mlflow.user':user_profile_name})
    
    # Train the model for different max_depth values
    for index, depth in enumerate([2, 5, 10, 15, 20]):
        hyperparams["max_depth"] = depth
        print(f"Fit estimator with max_depth={depth}")
    
        with mlflow.start_run(
            run_name=f"max_depth={depth}",
            description=f"Fit estimator with max_depth={depth}",
            nested=True
        ) as child_run:
            mlflow.set_tags({'mlflow.user':user_profile_name})
            
            mlflow.xgboost.autolog(silent=True)
            
            # Train the model
            model = xgb.train(
                params=hyperparams, 
                dtrain=dtrain, 
                evals = [(dtrain,'train'), (dtest,'eval')], 
                num_boost_round=num_boost_round, 
                early_stopping_rounds=early_stopping_rounds, 
                verbose_eval = 0
            )
    
            # Calculate metrics
            test_auc = roc_auc_score(test_label, model.predict(dtest))
            train_auc = roc_auc_score(train_label, model.predict(dtrain))
            
            # Log hyperparameters and metrics to the run
            mlflow.log_params(hyperparams)
            mlflow.log_metrics({"test_auc":test_auc, "train_auc":train_auc}, step=index)
    
            if test_auc > max_metric:
                best_model_run_id = child_run.info.run_id
                max_metric = test_auc
    
            print(f"Test AUC: {test_auc:.4f} | Train AUC: {train_auc:.4f}")

### The prototype is not registered, and that is deliberate

MLflow can act as a [model registry](https://mlflow.org/docs/latest/model-registry.html), and because the app runs with `AutoModelRegistrationEnabled`, every `mlflow.register_model()` call also creates a Model Package in the [SageMaker Model Registry](https://docs.aws.amazon.com/sagemaker/latest/dg/model-registry.html). Registration is therefore a **governance action**: whatever lands in the registry is a candidate for approval and deployment.

A kernel-trained model does not qualify: not reproducible, no training-job lineage, no inference specification. Part 1 stops at tracked experiments; Part 2 registers exactly one model, from the managed training job, with an inference specification attached.

For now, note which local run won the sweep.

In [ ]:
print(f"Best local run: {best_model_run_id} (test AUC {max_metric:.4f})")
print("Tracked, not registered — Part 2 registers the model trained by the SageMaker training job.")

### Compare the local runs in the MLflow UI
Open the MLflow UI on the experiment you just populated and compare the `max_depth` runs in **Chart view**.

In [ ]:
sm_client = boto3.Session().client("sagemaker")

experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

# get the presigned url to open the MLflow UI
presigned_url = sm_client.create_presigned_mlflow_app_url(
    Arn=mlflow_arn,
    ExpiresInSeconds=60,
    SessionExpirationDurationInSeconds=1800
)['AuthorizedUrl']

compare_run_link = f"{presigned_url.split('/auth')[0]}/#/experiments/{experiment_id}/runs?workspace=default&compareRunsMode=CHART"

In [ ]:
# first open the MLflow UI - you can close a new opened window
display(Javascript('window.open("{}");'.format(presigned_url)))

In [ ]:
display(Javascript('window.open("{}");'.format(compare_run_link)))

## Explore experiments with the MLflow UI
Open the app from the Studio UI: **MLflow** widget > **Open MLflow**.

![](img/mlflow-apps-studio.png)

The UI opens in a new browser window. Select your workspace (*default*) and the experiment generated above to view its runs.

![](img/experiment-mlflow.png)

**Chart view** compares runs; the eye icons show or hide individual runs.

![](img/comparing-runs.png)

The **Models** tab lists registered models and their versions.

![](img/logged-models.png)

---
# Part 2 — Scale with SageMaker AI jobs

Everything above ran in this kernel: fine for prototyping on a subset, but it does not scale, is not reproducible, and cannot be scheduled. Now move the identical logic into SageMaker managed jobs.

## Process data with SageMaker processing jobs
[SageMaker Processing](https://docs.aws.amazon.com/sagemaker/latest/dg/processing-job.html) runs a Python script on a chosen [processor class](https://sagemaker.readthedocs.io/en/stable/api/sagemaker_core.html#module-sagemaker.core.processing): it loads input from S3, runs the script in a container (built-in or custom), and uploads the output back to S3. SageMaker provisions the cluster for the job's duration and tears it down on completion.

![](img/sagemaker-processing.png)

### Why there is no separate local-mode processing run
[Local mode](https://docs.aws.amazon.com/sagemaker/latest/dg/studio-updated-local.html) runs a SageMaker job in a container on this JupyterLab space, useful for debugging a script before submitting it to managed compute. You already validated this preprocessing logic in Part 1, and a local-mode job would write the same outputs to the same S3 prefixes as the managed job below, so it is skipped to avoid duplicate work.

Local mode is still demonstrated once, under **Smoke-test with `ModelTrainer` in local container mode** below. It works with BYOC and most built-in containers, though some built-in algorithm containers such as XGBoost do not support it.

Create a folder for the processing code and requirements. The BYOC image already includes mlflow and the MLflow SageMaker plugin, so this `requirements.txt` is a no-op for that image. It is kept to demonstrate the per-job dependency mechanism `FrameworkProcessor` uses, so the code still works with a built-in image that does not bundle them.

In [ ]:
local_processing_dir = "./processing/"
os.makedirs(local_processing_dir, exist_ok=True)
with open(os.path.join(local_processing_dir, "requirements.txt"), "w") as file_handle:
    file_handle.write("mlflow>=3,<4\nsagemaker-mlflow>=0.5.0,<1\n")

#### Create the processing script
Move the Part 1 feature engineering into a `.py` file and add MLflow tracking. Two things are added beyond the notebook version:

- the test set is written as separate `test_x.csv` / `test_y.csv` files, which the endpoint test and the notebook 03 pipeline expect;
- the full feature set without the label is written as `baseline.csv`, the data baseline used by monitoring in notebook 04.

In [ ]:
%%writefile processing/preprocessing.py
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import pandas as pd
import numpy as np
import argparse
import os

from time import gmtime, strftime, sleep
import traceback

user_profile_name = os.getenv('USER')
print(os.environ)
def _parse_args():
    
    parser = argparse.ArgumentParser()
    # Data, model, and output directories
    # model_dir is always passed in from SageMaker. By default this is a S3 path under the default bucket.
    parser.add_argument('--filepath', type=str, default='/opt/ml/processing/input/')
    parser.add_argument('--filename', type=str, default='bank-additional-full.csv')
    parser.add_argument('--outputpath', type=str, default='/opt/ml/processing/output/')
    
    return parser.parse_known_args()

def process_data(df_data):
    # Indicator variable to capture when pdays takes a value of 999
    df_data["no_previous_contact"] = np.where(df_data["pdays"] == 999, 1, 0)

    # Indicator for individuals not actively employed
    df_data["not_working"] = np.where(
        np.isin(df_data["job"], ["student", "retired", "unemployed"]), 1, 0
    )

    # remove unnecessary data
    df_model_data = df_data.drop(
        ["duration", "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"],
        axis=1,
    )

    bins = [18, 30, 40, 50, 60, 70, 90]
    labels = ['18-29', '30-39', '40-49', '50-59', '60-69', '70-plus']

    df_model_data['age_range'] = pd.cut(df_model_data.age, bins, labels=labels, include_lowest=True)
    df_model_data = pd.concat([df_model_data, pd.get_dummies(df_model_data['age_range'], prefix='age', dtype=int)], axis=1)
    df_model_data.drop('age', axis=1, inplace=True)
    df_model_data.drop('age_range', axis=1, inplace=True)

    scaled_features = ['pdays', 'previous', 'campaign']
    df_model_data[scaled_features] = MinMaxScaler().fit_transform(df_model_data[scaled_features])

    df_model_data = pd.get_dummies(df_model_data, dtype=int)  # Convert categorical variables to sets of indicators

    # Replace "y_no" and "y_yes" with a single label column, and bring it to the front:
    df_model_data = pd.concat(
        [
            df_model_data["y_yes"].rename(target_col),
            df_model_data.drop(["y_no", "y_yes"], axis=1),
        ],
        axis=1,
    )
    
    return df_model_data

if __name__=="__main__":
    # Process arguments
    args, _ = _parse_args()
    
    target_col = "y"
    
    import mlflow
    
    # Set the Tracking Server URI using the ARN of the Tracking Server you created
    mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_ARN'])
    
    # Enable autologging in MLflow
    mlflow.autolog()

    # Use the active run_id to log 
    with mlflow.start_run(run_id=os.environ['MLFLOW_RUN_ID']) as run:
        # process data
        df_model_data = process_data(pd.read_csv(os.path.join(args.filepath, args.filename), sep=";"))
    
        # Shuffle and splitting dataset
        df_shuffled = df_model_data.sample(frac=1, random_state=1729)
        train_end = int(0.7 * len(df_model_data))
        validation_end = int(0.9 * len(df_model_data))
        train_data = df_shuffled.iloc[:train_end]
        validation_data = df_shuffled.iloc[train_end:validation_end]
        test_data = df_shuffled.iloc[validation_end:]
    
        print(f"Data split > train:{train_data.shape} | validation:{validation_data.shape} | test:{test_data.shape}")
        mlflow.log_params(
            {
                "train": train_data.shape,
                "validate": validation_data.shape,
                "test": test_data.shape
            }
        )

        mlflow.set_tags(
            {
                'mlflow.user':user_profile_name,
                'mlflow.source.type':'JOB'
            }
        )
        
        # Save datasets locally
        train_data.to_csv(os.path.join(args.outputpath, 'train/train.csv'), index=False, header=False)
        validation_data.to_csv(os.path.join(args.outputpath, 'validation/validation.csv'), index=False, header=False)
        test_data[target_col].to_csv(os.path.join(args.outputpath, 'test/test_y.csv'), index=False, header=False)
        test_data.drop([target_col], axis=1).to_csv(os.path.join(args.outputpath, 'test/test_x.csv'), index=False, header=False)
        
        # Save the baseline dataset for model monitoring
        df_model_data.drop([target_col], axis=1).to_csv(os.path.join(args.outputpath, 'baseline/baseline.csv'), index=False, header=False)

        mlflow.log_artifact(local_path=os.path.join(args.outputpath, 'baseline/baseline.csv'))

        mlflow.log_artifact(local_path=os.path.join(args.outputpath, "train/train.csv"))
        mlflow.log_artifact(local_path=os.path.join(args.outputpath, "validation/validation.csv"))
        mlflow.log_artifact(local_path=os.path.join(args.outputpath, "test/test_x.csv"))
        mlflow.log_artifact(local_path=os.path.join(args.outputpath, "test/test_y.csv"))
    
    print("## Processing complete. Exiting.")


#### Create an experiment run
Create a new run to track the processing job's parameters, configuration, inputs, and outputs.

In [ ]:
run_name = name_from_base("container-processing-", short=True)
run_id = mlflow.start_run(
    run_name=run_name,
    description="feature engineering with a SageMaker processing job (Part 2)").info.run_id

#### Set the job inputs and outputs
SageMaker maps your data to local paths on the processing container's EBS volume:

![](img/data-processing.png)

Input can come from a local directory, S3, Athena, or Redshift. The managed job reads the raw dataset from S3 (`s3_data_uri`, uploaded in notebook 00) and writes the processed datasets back to S3. Define the S3 output locations first; they are stored with `%store` because notebooks 03 and 04 consume them.

In [ ]:
train_s3_url = s3_path_join("s3://", bucket_name, bucket_prefix, "train")
validation_s3_url = s3_path_join("s3://", bucket_name, bucket_prefix, "validation")
test_s3_url = s3_path_join("s3://", bucket_name, bucket_prefix, "test")
baseline_s3_url = s3_path_join("s3://", bucket_name, bucket_prefix, "baseline")

%store train_s3_url
%store validation_s3_url
%store test_s3_url
%store baseline_s3_url

Define the `ProcessingInput` and `ProcessingOutput` objects that tell SageMaker where to find the input data and where to save the output.

In [ ]:
processing_inputs = [
    ProcessingInput(
        input_name="data",
        s3_input=ProcessingS3Input(
            s3_uri=s3_data_uri,
            local_path="/opt/ml/processing/input",
            s3_data_type="S3Prefix",
            s3_input_mode="File")
    )
]

In [ ]:
processing_outputs = [
    ProcessingOutput(
        output_name="train",
        s3_output=ProcessingS3Output(
            s3_uri=train_s3_url,
            local_path="/opt/ml/processing/output/train",
            s3_upload_mode="EndOfJob")
    ),
    ProcessingOutput(
        output_name="validation",
        s3_output=ProcessingS3Output(
            s3_uri=validation_s3_url,
            local_path="/opt/ml/processing/output/validation",
            s3_upload_mode="EndOfJob")
    ),
    ProcessingOutput(
        output_name="test",
        s3_output=ProcessingS3Output(
            s3_uri=test_s3_url,
            local_path="/opt/ml/processing/output/test",
            s3_upload_mode="EndOfJob")
    ),
    ProcessingOutput(
        output_name="baseline",
        s3_output=ProcessingS3Output(
            s3_uri=baseline_s3_url,
            local_path="/opt/ml/processing/output/baseline",
            s3_upload_mode="EndOfJob")
    )
]

#### Create the processor
Instantiate a [`FrameworkProcessor`](https://docs.aws.amazon.com/sagemaker/latest/dg/processing-job-frameworks.html) with the BYOC container from notebook 00 on an `ml.m5.xlarge` instance.

`FrameworkProcessor` overrides the container entrypoint with a generated `runproc.sh`, so any image providing `bash`, `tar`, `pip`, `python3`, and the script's dependencies works. The MLflow coordinates are passed as environment variables so the script logs into the run you just opened.

In [ ]:
# Use the custom BYOC image built in notebook 00 (restored via %store -r).
# It satisfies the FrameworkProcessor container contract: bash, tar, pip,
# and python3, plus the sklearn/pandas/mlflow stack used by preprocessing.py.
print(f"Processing image: {xgboost_image}")

processor = FrameworkProcessor(
    image_uri=xgboost_image,
    role=sm_role,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    max_runtime_in_seconds=600,  # stop the job (and billing) after 10 minutes
    base_job_name='from-idea-to-prod-processing',
    command=["python3"],
    env={
        'MLFLOW_TRACKING_ARN': mlflow_arn,
        'MLFLOW_EXPERIMENT_NAME': experiment_name,
        'MLFLOW_RUN_ID': run_id,
        'REGION': region,
        'USER': user_profile_name,
    }
)

#### Start the SageMaker processing job
On its first run the processor authenticates Docker to ECR and pulls the BYOC image, so this run is slower than later ones.

In [ ]:
processor.run(
    code="preprocessing.py",
    source_dir="./processing",
    requirements="requirements.txt",
    inputs=processing_inputs,
    outputs=processing_outputs,
    wait=False,
    logs=False,
)

# The processing job above was submitted with wait=False, so .run() returned
# immediately without blocking. Poll the processing job status in a loop here
# until it reaches a terminal state.
processing_job = processor.latest_job
print(f"Polling processing job: {processing_job.processing_job_name}")

terminal_states = {"Completed", "Failed", "Stopped"}
while True:
    processing_job.refresh()
    status = processing_job.processing_job_status
    print(f"  {time.strftime('%H:%M:%S')} | status={status}")
    if status in terminal_states:
        break
    time.sleep(30)

if status != "Completed":
    failure_reason = processing_job.failure_reason or ""
    if "MaxRuntimeExceeded" in failure_reason:
        # Rescue path: the job hit the max_runtime_in_seconds cap (600s).
        # SageMaker already stopped the containers and billing. Close the
        # MLflow run with a failed state and give actionable guidance
        # instead of a bare stack trace.
        mlflow.end_run(status="FAILED")
        raise TimeoutError(
            f"Processing job {processing_job.processing_job_name} exceeded its "
            f"maximum runtime of 600 seconds and was stopped by SageMaker "
            f"(FailureReason: {failure_reason}). If the job legitimately needs "
            f"more time, increase max_runtime_in_seconds on the FrameworkProcessor; "
            f"otherwise check the CloudWatch logs of the job for a hang "
            f"(e.g. waiting on a network resource such as the MLflow tracking server)."
        )
    mlflow.end_run(status="FAILED")
    raise RuntimeError(
        f"Processing job {processing_job.processing_job_name} ended with status '{status}': "
        f"{failure_reason}"
    )
print(f"Processing job {processing_job.processing_job_name} completed successfully.")

mlflow.end_run()

Once the job finishes, read the training output back from S3 as a quick sanity check.

In [ ]:
job = processor.latest_job
output_uris = {
    output.output_name: output.s3_output.s3_uri
    for output in job.processing_output_config.outputs
}

# Sanity check on the train channel only — the other outputs are consumed by the
# training job, the endpoint test, and notebooks 03/04 straight from S3.
train_data = pd.read_csv(
    StringIO(S3Downloader.read_file(S3Downloader.list(output_uris["train"])[0])),
    header=None,
)
train_data.head()

## Model training with SageMaker training jobs
Run training as a [SageMaker training job](https://docs.aws.amazon.com/sagemaker/latest/dg/train-model.html), following the same approach as processing: provide a training image, source code, and compute configuration.

![](img/sagemaker-training-options.png)

Create a folder for the training scripts with a `requirements.txt` for MLflow.

In [ ]:
local_training_dir = "./training/"
os.makedirs(local_training_dir, exist_ok=True)
with open(os.path.join(local_training_dir, "requirements.txt"), "w") as file_handle:
    file_handle.write("sagemaker>=3,<4")

### Prepare data and hyperparameters
Define the data input channels for the training job.

In [ ]:
training_input_data = [
    InputData(channel_name='train', data_source=train_s3_url),
    InputData(channel_name='validation', data_source=validation_s3_url),
]

output_s3_url = f"s3://{bucket_name}/{bucket_prefix}/output"
%store output_s3_url

Set the instance type and count for the training jobs.

In [ ]:
train_instance_count = 1
train_instance_type = "ml.r5.xlarge"

Prepare the hyperparameters and MLflow settings as environment variables.

In [ ]:
hyperparams = {
    'num_round': 50,
    'max_depth': 3,
    'eta': 0.5,
    'alpha': 2.5,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'early_stopping_rounds': 10,
    'verbosity': 1
}

In [ ]:
env_variables = {
    'MLFLOW_TRACKING_ARN': mlflow_arn,
    'MLFLOW_EXPERIMENT_NAME': experiment_name,
    'USER': user_profile_name,
    'REGION': region,
    'TRAINING_JOB_CONTAINER': xgboost_image
}

print(env_variables)

### Train using the `@remote` decorator
The [`@remote` decorator](https://docs.aws.amazon.com/sagemaker/latest/dg/train-remote-decorator-invocation.html) runs a decorated Python function as a SageMaker training job: it serializes the function and its arguments, provisions the instance, and runs the function there. Annotate the function and call it as usual.

#### Define the training function
An ordinary Python function with no SageMaker API inside. You validated this training logic locally in Part 1, so it goes straight to a managed job in the next cell.

In [ ]:
def train_xgboost(train_uri, validation_uri, hyperparams):
    """Train an XGBoost model"""
    import os
    import boto3
    import pandas as pd
    import xgboost as xgb
    from sklearn.metrics import roc_auc_score
    import mlflow
    from io import StringIO

    # Download data from S3
    s3 = boto3.client('s3')
    # print(os.environ)
    def read_csv_from_s3(s3_uri):
        bucket = s3_uri.split('/')[2]
        key = '/'.join(s3_uri.split('/')[3:])
        # List objects to find CSV files
        resp = s3.list_objects_v2(Bucket=bucket, Prefix=key)
        csv_files = [o['Key'] for o in resp.get('Contents', []) if o['Key'].endswith('.csv')]
        dfs = []
        for f in csv_files:
            obj = s3.get_object(Bucket=bucket, Key=f)
            dfs.append(pd.read_csv(StringIO(obj['Body'].read().decode('utf-8')), header=None))
        return pd.concat(dfs, ignore_index=True)

    train_df = read_csv_from_s3(train_uri)
    val_df = read_csv_from_s3(validation_uri)

    dtrain = xgb.DMatrix(train_df.iloc[:, 1:], label=train_df.iloc[:, 0])
    dval = xgb.DMatrix(val_df.iloc[:, 1:], label=val_df.iloc[:, 0])

    # Setup MLflow
    mlflow.set_tracking_uri(os.getenv('MLFLOW_TRACKING_ARN'))
    mlflow.set_experiment(experiment_name=os.getenv('MLFLOW_EXPERIMENT_NAME'))
    mlflow.xgboost.autolog(log_model_signatures=True, log_datasets=True, log_models=False)

    train_hp = {k: v for k, v in hyperparams.items() if k not in ('num_round', 'eval_metric', 'early_stopping_rounds', 'alpha', 'colsample_bytree')}

    with mlflow.start_run(run_name='remote-decorator-training', description='xgboost via @remote decorator') as run:
        booster = xgb.train(
            params=train_hp,
            dtrain=dtrain,
            evals=[(dtrain, 'train'), (dval, 'validation')],
            num_boost_round=hyperparams['num_round'],
        )
        val_auc = roc_auc_score(dval.get_label(), booster.predict(dval))
        train_auc = roc_auc_score(dtrain.get_label(), booster.predict(dtrain))
        mlflow.xgboost.log_model(booster, name='xgboost')
        mlflow.log_metrics({'validation_auc': val_auc, 'train_auc': train_auc})
        mlflow.set_tags({
            'mlflow.user': os.environ.get('USER', 'n/a'),
            'mlflow.source.type': 'JOB',
            'mlflow.source.name': os.environ.get('TRAINING_JOB_ARN', 'n/a'),
            'sagemaker.training_container_image': os.environ.get('TRAINING_JOB_CONTAINER', 'n/a'),
            'sagemaker.runtime': 'sagemaker',
            'sagemaker.training_job_arn': os.environ.get('TRAINING_JOB_ARN', 'n/a')
        })
        print(f'train-auc: {train_auc:.4f}, validation-auc: {val_auc:.4f}')

    return booster

In [ ]:
## set ENV variables locally
import os

for k, v in env_variables.items():
    os.environ[k] = v

In [ ]:
booster = train_xgboost(train_s3_url, validation_s3_url, hyperparams)

#### Annotate the function with the `@remote` decorator
To run on managed infrastructure, decorate a thin wrapper `remote_train_xgboost` with the same signature that calls `train_xgboost`, avoiding a redefinition of the body. The decorator arguments describe the job: instance type, role, extra dependencies, environment variables, and the container image.

In [ ]:
@remote(
    instance_type=train_instance_type,
    role=sm_role,
    dependencies='./training/requirements.txt',
    environment_variables=env_variables,
    job_name_prefix='from-idea-to-prod-remote',
    keep_alive_period_in_seconds=3600,
    image_uri=xgboost_image
)
def remote_train_xgboost(train_uri, validation_uri, hyperparams):
    """Train an XGBoost model using the @remote decorator."""
    return train_xgboost(train_uri, validation_uri, hyperparams)

In [ ]:
booster = remote_train_xgboost(train_s3_url, validation_s3_url, hyperparams)

---

`@remote` lifts a local function to a managed job with minimal changes. `ModelTrainer` gives more control over the training container, script packaging, and output artifacts, and is the same pattern used in SageMaker Pipelines.

### Train using `ModelTrainer` in script mode
The [`ModelTrainer`](https://sagemaker.readthedocs.io/en/stable/api/training/model_trainer.html) class gives full control over the training job: run a script inside a specific container (built-in or your own) with custom hyperparameters, input channels, and output locations. Prefer it for jobs at scale and for jobs in a pipeline.

The `@remote` decorator packaged the notebook environment plus a `requirements.txt`. [`ModelTrainer`](https://sagemaker.readthedocs.io/en/stable/training/index.html) instead takes a Docker image as its environment. You use the BYOC image from notebook 00, the same image that ran the processing job, so processing, training, and inference share one dependency set. (Alternatively, fetch the [pre-built XGBoost container](https://github.com/aws/sagemaker-xgboost-container).)

In [ ]:
xgb_training_image = xgboost_image
print(f'XGBoost training image: {xgb_training_image}')

#### Prepare the training script
Create a `train.py` with the training script.

In [ ]:
%%writefile ./training/train.py
import argparse
import json
import os
import pickle as pkl

import pandas as pd
from sklearn.metrics import roc_auc_score
import xgboost as xgb

# NOTE: The original script used `get_dmatrix` from `sagemaker_xgboost_container`,
# a module that only exists inside AWS's managed XGBoost container. The BYOC
# image built from `byoc/Dockerfile` does not include it. The local helper
# `load_csv_dmatrix` below is a drop-in replacement that uses only pandas and
# xgboost, so the script works against both the AWS-managed image and the BYOC
# image.
def load_csv_dmatrix(channel_dir):
    """Build an xgb.DMatrix from all CSV files in a SageMaker channel dir.
    Assumes header-less CSVs with the label in column 0 (SageMaker XGBoost convention)."""
    if channel_dir is None or not os.path.isdir(channel_dir):
        return None
    files = sorted(
        os.path.join(channel_dir, f)
        for f in os.listdir(channel_dir)
        if not f.startswith('.') and os.path.isfile(os.path.join(channel_dir, f))
    )
    if not files:
        return None
    frames = [pd.read_csv(f, header=None) for f in files]
    df = pd.concat(frames, ignore_index=True)
    labels = df.iloc[:, 0].values
    features = df.iloc[:, 1:].values
    return xgb.DMatrix(features, label=labels)

from time import gmtime, strftime

suffix = strftime('%d-%H-%M-%S', gmtime())
user_profile_name = os.getenv('USER', 'sagemaker')
experiment_name = os.getenv('MLFLOW_EXPERIMENT_NAME')
region = os.getenv('REGION')


def _xgb_train(params, dtrain, dval, evals, num_boost_round, model_dir, is_master):
    booster = xgb.train(params=params, dtrain=dtrain, evals=evals, num_boost_round=num_boost_round)

    val_auc = roc_auc_score(dval.get_label(), booster.predict(dval))
    train_auc = roc_auc_score(dtrain.get_label(), booster.predict(dtrain))
    mlflow.log_params(params)
    mlflow.log_metrics({'validation_auc': val_auc, 'train_auc': train_auc})
    print(f'[0]#011train-auc:{train_auc}#011validation-auc:{val_auc}')

    if is_master:
        model_location = model_dir + '/xgboost-model'
        pkl.dump(booster, open(model_location, 'wb'))
        print(f'Stored trained model at {model_location}')
    
    return booster


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--max_depth', type=int)
    parser.add_argument('--eta', type=float)
    parser.add_argument('--alpha', type=float)
    parser.add_argument('--gamma', type=int)
    parser.add_argument('--min_child_weight', type=float)
    parser.add_argument('--subsample', type=float)
    parser.add_argument('--colsample_bytree', type=float)
    parser.add_argument('--verbosity', type=int)
    parser.add_argument('--objective', type=str)
    parser.add_argument('--num_round', type=int)
    parser.add_argument('--early_stopping_rounds', type=int)
    parser.add_argument('--tree_method', type=str, default='auto')
    parser.add_argument('--predictor', type=str, default='auto')
    parser.add_argument('--output_data_dir', type=str, default=os.environ.get('SM_OUTPUT_DATA_DIR'))
    parser.add_argument('--model_dir', type=str, default=os.environ.get('SM_MODEL_DIR'))
    parser.add_argument('--train', type=str, default=os.environ.get('SM_CHANNEL_TRAIN'))
    parser.add_argument('--validation', type=str, default=os.environ.get('SM_CHANNEL_VALIDATION'))
    parser.add_argument('--sm_hosts', type=str, default=os.environ.get('SM_HOSTS'))
    parser.add_argument('--sm_current_host', type=str, default=os.environ.get('SM_CURRENT_HOST'))
    parser.add_argument('--sm_training_env', type=str, default=os.environ.get('SM_TRAINING_ENV'))

    args, _ = parser.parse_known_args()

    sm_hosts = json.loads(args.sm_hosts)
    sm_current_host = args.sm_current_host
    dtrain = load_csv_dmatrix(args.train)
    dval = load_csv_dmatrix(args.validation)
    watchlist = [(dtrain, 'train'), (dval, 'validation')] if dval is not None else [(dtrain, 'train')]
    sm_training_env = json.loads(args.sm_training_env)

    import mlflow
    mlflow.set_tracking_uri(os.getenv('MLFLOW_TRACKING_ARN'))
    mlflow.set_experiment(experiment_name=experiment_name if experiment_name else f'train-{suffix}')
    mlflow.xgboost.autolog(log_model_signatures=True, log_datasets=True, log_models=False)

    train_hp = {
        'max_depth': args.max_depth, 'eta': args.eta, 'gamma': args.gamma,
        'min_child_weight': args.min_child_weight, 'subsample': args.subsample,
        'verbosity': args.verbosity, 'objective': args.objective,
        'tree_method': args.tree_method, 'predictor': args.predictor,
    }

    xgb_train_args = dict(
        params=train_hp, dtrain=dtrain, dval=dval, evals=watchlist,
        num_boost_round=args.num_round, model_dir=args.model_dir,
    )

    # Resume parent run if MLFLOW_RUN_ID is set (for nested run grouping)
    parent_run_id = os.getenv('MLFLOW_RUN_ID')
    if parent_run_id:
        mlflow.start_run(run_id=parent_run_id)

    with mlflow.start_run(
        run_name=f'container-training-{suffix}',
        description='xgboost running in SageMaker container in script mode',
        nested=True if parent_run_id else False,
    ) as run:
        mlflow.set_tags({
            'mlflow.user': user_profile_name,
            'mlflow.source.type': 'JOB',
            'mlflow.source.name': os.environ.get('TRAINING_JOB_ARN', 'n/a'),
            'sagemaker.training_container_image': os.environ.get('TRAINING_JOB_CONTAINER', 'n/a'),
            'sagemaker.job_name': sm_training_env['job_name'],
            'sagemaker.runtime': 'sagemaker',
            'sagemaker.training_job_arn': os.environ.get('TRAINING_JOB_ARN', 'n/a')
        })
        if dtrain:
            xgb_train_args['is_master'] = True
            booster = _xgb_train(**xgb_train_args)
            mlflow.xgboost.log_model(booster, name=f'xgboost-{suffix}')
        else:
            raise ValueError('Training channel must have data to train model.')


#### Smoke-test with `ModelTrainer` in local container mode
`ModelTrainer` supports `LOCAL_CONTAINER` and `SAGEMAKER_TRAINING_JOB` modes. This is the one place in the notebook that uses **local mode** as a smoke test: the script runs inside the BYOC image in a Docker container on this space, validating the container contract (`SM_CHANNEL_*` paths, `SM_MODEL_DIR`, hyperparameter parsing, installed dependencies) in seconds instead of waiting on a managed instance.

In [ ]:
train_locally = ModelTrainer(
    training_image=xgb_training_image,
    source_code=SourceCode(
        source_dir="./training",
        entry_script="train.py",
    ),
    compute=Compute(instance_type="local", instance_count=1),
    training_mode=TrainingMode.LOCAL_CONTAINER,
    hyperparameters=hyperparams,
    base_job_name='from-idea-to-prod-local',
    environment=env_variables,
)

<div class="alert alert-info">If you get an error <code>No such file or directory: 'docker'</code> in the code below, you need to run the section <b>Install Docker to enable Studio local mode</b> from the notebook 00.</div>

In [ ]:
try:
    train_locally.train(
        input_data_config=[
            InputData(channel_name="train", data_source=train_s3_url, content_type="text/csv"),
            InputData(channel_name="validation", data_source=validation_s3_url, content_type="text/csv"),
        ],
        wait=True,
        logs=True,
    )
except PermissionError as e:
    if "__pycache__" in str(e):
        print(f"⚠️ Training succeeded but cleanup failed (harmless): {e}")
    else:
        raise

#### Train on SageMaker managed infrastructure
The only changes from the smoke test are the compute configuration, `SAGEMAKER_TRAINING_JOB` mode, and an explicit output location. The script is unchanged.

In [ ]:
trainer_managed = ModelTrainer(
    training_image=xgb_training_image,
    source_code=SourceCode(
        source_dir="./training",
        entry_script="train.py",
    ),
    compute=Compute(
        instance_type=train_instance_type,
        instance_count=train_instance_count
    ),
    hyperparameters=hyperparams,
    base_job_name='from-idea-to-prod-training',
    training_mode=TrainingMode.SAGEMAKER_TRAINING_JOB,
    output_data_config=OutputDataConfig(s3_output_path=output_s3_url),
    environment=env_variables,
)

In [ ]:
trainer_managed.train(
    input_data_config=training_input_data,
    wait=False,
    logs=False,
)

In [ ]:
# The training job above was submitted with wait=False, so .train() returned
# immediately without blocking. Poll the training job status in a loop here
# until it reaches a terminal state.
training_job = trainer_managed._latest_training_job
print(f"Polling training job: {training_job.training_job_name}")

terminal_states = {"Completed", "Failed", "Stopped"}
while True:
    training_job.refresh()
    status = training_job.training_job_status
    print(f"  {time.strftime('%H:%M:%S')} | status={status} | secondary={training_job.secondary_status}")
    if status in terminal_states:
        break
    time.sleep(30)

if status != "Completed":
    raise RuntimeError(
        f"Training job {training_job.training_job_name} ended with status '{status}': "
        f"{training_job.failure_reason}"
    )
print(f"Training job {training_job.training_job_name} completed successfully.")

After the training job completes, fetch the output artifacts.

In [ ]:
training_job = trainer_managed._latest_training_job
print(f"Training job: {training_job.training_job_name}")
print(f"Model artifacts: {training_job.model_artifacts.s3_model_artifacts}")

In [ ]:
display(
    HTML('<b>See the SageMaker <a target="top" href="https://{}.console.aws.amazon.com/sagemaker/home?region={}#/jobs/{}">training job</a></b>'.format(
            region, region, trainer_managed._latest_training_job.training_job_name))
)

## Select the best candidate model

The experiment now holds several **candidate** models, each logged by a SageMaker job:

| Candidate | Logged by |
|---|---|
| `remote-decorator-training` | the `@remote` training job |
| `container-training-…` | the `ModelTrainer` `LOCAL_CONTAINER` smoke test |
| `container-training-…` | the `ModelTrainer` managed training job |

Rather than hard-code which to register, ask the tracking server which performed best. [`mlflow.search_logged_models()`](https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.search_logged_models) returns MLflow 3 `LoggedModel` entities, each carrying its `model_id`, `artifact_location`, `source_run_id`, and metrics. `train.py` logs the model with `mlflow.xgboost.log_model(booster, name="xgboost")`, which makes it searchable with the run's metrics attached.

Ordering by `metrics.validation_auc` excludes the Part 1 prototypes, which logged `test_auc`. The two `container-training` candidates run the same script on the same data, so their scores are effectively identical and the tie is broken arbitrarily. To restrict the search to managed training jobs, add a tag filter on `sagemaker.job_name`, which `train.py` sets.

In [ ]:
mlflow_client = MlflowClient()

metric_name = "validation_auc"
experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

candidates = mlflow.search_logged_models(
    experiment_ids=[experiment_id],
    filter_string=f"metrics.{metric_name} > 0",
    order_by=[{"field_name": f"metrics.{metric_name}", "ascending": False}],
    output_format="list",
)

if not candidates:
    raise RuntimeError(
        f"No logged model in experiment '{experiment_name}' has a '{metric_name}' metric. "
        "Did the training jobs above complete and log their models?"
    )


def metric_of(model, key):
    return {m.key: m.value for m in (model.metrics or [])}.get(key)


print(f"{'rank':<5}{'run name':<34}{metric_name:>12}   model id")
print("-" * 100)
for rank, candidate in enumerate(candidates, 1):
    run_name = mlflow_client.get_run(candidate.source_run_id).info.run_name
    print(f"{rank:<5}{run_name:<34}{metric_of(candidate, metric_name):>12.4f}   {candidate.model_id}")

logged_model = candidates[0]
best_run_id = logged_model.source_run_id

print(f"\nSelected model id: {logged_model.model_id}")
print(f"Source run:        {best_run_id}")
print(f"Artifact location: {logged_model.artifact_location}")
print(f"{metric_name}:     {metric_of(logged_model, metric_name):.4f}")

## Explain the model before you register it

Approval needs more than one AUC number. Before registration, produce a **feature-attribution and fairness report** for exactly the model you are about to register. Logging it here, before `register_model()`, onto the same MLflow run as the model means the Model Package inherits lineage that already includes the explanation.

### What computes the explanation

The report uses **SHAP** for feature attribution and standardized **fairness metrics** (class imbalance, difference in positive proportions, disparate impact) computed with pandas. The logic lives in [`pipeline_steps/explain.py`](pipeline_steps/explain.py) as a single plain Python function, `explain(...)`. You run it here as a **managed job** with `@remote`; notebook 03 wraps the same function with `@step` as a pipeline step, reusing one implementation.

The function uses `shap.TreeExplainer`, not `KernelExplainer`. Kernel SHAP is model-agnostic, for opaque pipelines; our model is a plain XGBoost booster, so tree SHAP applies, is **exact** (no sampling), and is fast enough to explain the entire test set.

### Prepare the test data
Download the held-out test set the processing job produced, to check that the feature names line up with it. Notebook 04 downloads the same files to test the endpoint.

In [ ]:
!aws s3 cp $test_s3_url/test_x.csv tmp/test_x.csv
!aws s3 cp $test_s3_url/test_y.csv tmp/test_y.csv

Load it into memory. The explainability job reads the same files straight from S3; these local copies are for the checks in this notebook.

In [ ]:
test_x = np.loadtxt("tmp/test_x.csv", delimiter=",")
test_y = np.loadtxt("tmp/test_y.csv", delimiter=",")

print(f"test_x: {test_x.shape} | test_y: {test_y.shape}")

### Feature names and the fairness facet

The processing job wrote header-less CSVs, so the feature names are not in `test_x.csv`. They come from the dataframe built in Part 1 (the same one-hot columns, in the same order) and travel to the job as an argument.

Fairness analysis needs a **facet**: a group whose treatment is compared against everyone else. Part 1's age binning left one-hot columns per age band, so `age_60-69` and `age_70-plus` together express *customer aged 60 or over*, a legitimate question for a term-deposit marketing model.

In [ ]:
# Part 1 built df_model_data with the target first, then the one-hot features, and
# preprocessing.py applied the identical transformation before writing test_x.csv.
feature_names = [c for c in df_model_data.columns if c != target_col]

facet_columns = ["age_60-69", "age_70-plus"]

assert len(feature_names) == test_x.shape[1], (
    f"{len(feature_names)} feature names vs {test_x.shape[1]} columns in test_x.csv"
)
missing_facets = [c for c in facet_columns if c not in feature_names]
assert not missing_facets, f"facet columns not in the feature set: {missing_facets}"

print(f"{len(feature_names)} features, e.g. {feature_names[:6]}")
print(f"Fairness facet: {facet_columns}")

# Notebook 04 names the columns of the header-less baseline and captured data with these.
%store feature_names

In [ ]:
%%writefile ./explainability/requirements.txt
# The BYOC image already ships xgboost, pandas, numpy, mlflow and s3fs.
# The explainability job only needs these two extra libraries.
shap
matplotlib

### Run the report as a managed job

Apply `@remote` to the imported `explain` function. Two arguments matter beyond the usual compute settings:

- **`dependencies`**: the BYOC image ships xgboost, pandas, mlflow, and s3fs, but not `shap` or `matplotlib`. [`explainability/requirements.txt`](explainability/requirements.txt) adds those two, and the decorator pip-installs them into the job.
- **`include_local_workdir`**: `explain` lives in a module, so the job needs the `pipeline_steps/` directory shipped alongside it. `CustomFileFilter` keeps the upload small by excluding datasets, notebooks, and images.

The environment variables follow the notebook 03 pipeline-step convention: `explain` lets MLflow pick up `MLFLOW_TRACKING_URI` and `MLFLOW_EXPERIMENT_NAME` from the environment rather than calling `set_tracking_uri` itself.

In [ ]:
explain_env = {
    # MLflow reads these two natively from the environment.
    "MLFLOW_TRACKING_URI": mlflow_arn,
    "MLFLOW_EXPERIMENT_NAME": experiment_name,
    # picked up by pipeline_steps/runtime_utils.log_runtime_info()
    "CONTAINER_IMAGE": xgboost_image,
    "USER": user_profile_name,
    "REGION": region,
}

# Applying the decorator as a call, rather than with @-syntax, keeps the shared
# function untouched - and reads exactly like step(explain, ...) does in notebook 03.
remote_explain = remote(
    instance_type=train_instance_type,
    role=sm_role,
    image_uri=xgboost_image,
    dependencies="./explainability/requirements.txt",
    include_local_workdir=True,
    custom_file_filter=CustomFileFilter(
        ignore_name_patterns=["data", "tmp", "img", "*.ipynb", "__pycache__", ".git"]
    ),
    environment_variables=explain_env,
    job_name_prefix="from-idea-to-prod-explain",
    keep_alive_period_in_seconds=3600,
)(explain)

explain_result = remote_explain(
    mlflow_run_id=best_run_id,
    test_x_data_s3_path=f"{test_s3_url}/test_x.csv",
    test_y_data_s3_path=f"{test_s3_url}/test_y.csv",
    feature_names=feature_names,
    output_s3_prefix=f"s3://{bucket_name}/{bucket_prefix}",
    facet_columns=facet_columns,
)

### Read the report

The job logged the plots, the per-instance SHAP table, and the bias metrics to the model's MLflow run, and returned the headline numbers. Open the run in MLflow for the beeswarm, the global importance bar chart, and the waterfall for the most confident prediction.

Read `DI_disparate_impact` as the ratio of positive-prediction rates between the facet and the comparison group: `1.0` is parity, and the further from `1.0`, the more differently the model treats the two groups. `DPL` versus `DPPL` tells you whether a gap was already in the labels (pre-training) or introduced by the model (post-training).

Check `n_facet_group` before trusting either ratio: the age-60+ facet is only a few dozen of the ~4,100 test rows, so its rates move a lot with a handful of predictions. A large `DI` on a small facet is a prompt to collect more data for that group, not evidence of a broken model.

In [ ]:
print(f"Explained model:  {explain_result['explained_model_id']}")
print(f"E[y] (log-odds):  {explain_result['shap_expected_value_logodds']:.4f}")
print(f"Report in S3:     {explain_result['report_s3_prefix']}")

print("\nTop features by mean |SHAP|:")
for rank, feature in enumerate(explain_result["top_features"][:10], 1):
    print(f"  {rank:<3}{feature}")

print("\nFairness metrics for the age-60+ facet:")
for name, value in explain_result["bias_metrics"].items():
    print(f"  {name:<40}{value:.4f}" if isinstance(value, float) else f"  {name:<40}{value}")

## Explore the explainability results

In [ ]:
explain_run_link = (
    f"{presigned_url.split('/auth')[0]}"
    f"/#/experiments/{experiment_id}/runs/{explain_result['explainability_run_id']}?workspace=default"
)
display(Javascript('window.open("{}");'.format(explain_run_link)))

## Explore the explainability artifacts

In [ ]:
explain_run_link = (
    f"{presigned_url.split('/auth')[0]}"
    f"/#/experiments/{experiment_id}/runs/{explain_result['explainability_run_id']}/artifacts?workspace=default"
)
display(Javascript('window.open("{}");'.format(explain_run_link)))

### Make the model deployable, then register it

A registered model is only useful if it can be *served*. The **inference-specification logging** pattern makes the logged model deployable:

1. Upload an `inference.py` into the logged model's artifact store under `code/`, with `MlflowClient.log_model_artifacts()`.
2. Log an **inference specification** on the logged model with the `sagemaker-mlflow` plugin (`>= 0.5.0`): the serving image, an `S3Prefix` `ModelDataSource` pointing at the artifact location, and the `SAGEMAKER_PROGRAM` / `SAGEMAKER_SUBMIT_DIRECTORY` environment variables.
3. Call `mlflow.register_model()`.

```
MLflow artifact store ◀────────── serves directly from ──────────┐
        │  log_inference_specification()                         │
        ▼                                                        │
MLflow ──register──▶ auto-sync ──▶ Model Package (deployable) ──▶ Endpoint
```

The artifact store stays the single source of truth for the model bytes: no repacking, no copies, no `update_model_package` to patch the spec afterward.

<div class="alert alert-warning">
<b>Order matters:</b> log the specification <b>before</b> <code>mlflow.register_model()</code>. The auto-sync copies the spec onto the Model Package at registration time; log it afterward and the package is already created without it.
</div>

The reusable [`pipeline_steps/register.py`](pipeline_steps/register.py) step does exactly this in notebook 03. Here you do it inline, to see each piece.

#### Log an `inference.py` into the model's artifact store

The BYOC image serves with its own FastAPI stack (see [`byoc/serving_app.py`](byoc/serving_app.py)), which loads the script named by `SAGEMAKER_PROGRAM` and dispatches to `model_fn` / `input_fn` / `predict_fn` / `output_fn`. It has no default model loader, so a `model_fn` is mandatory: without it the container fails every `/ping` health check and the endpoint never reaches `InService`.

The `S3Prefix` `ModelDataSource` downloads everything under the artifact location to `/opt/ml/model`, so `code/inference.py` arrives at `/opt/ml/model/code/inference.py` and the MLflow model files (`MLmodel`, the booster, `conda.yaml`) land at `/opt/ml/model`, which is why `model_fn` can call `mlflow.xgboost.load_model(model_dir)`.

In [ ]:
INFERENCE_SCRIPT = r"""
import json

import numpy as np


def model_fn(model_dir):
    # The S3Prefix ModelDataSource downloaded the whole MLflow model directory
    # here, so MLflow can load the booster and resolve its serialization format.
    import mlflow.xgboost

    return mlflow.xgboost.load_model(model_dir)


def input_fn(request_body, request_content_type):
    if isinstance(request_body, (bytes, bytearray)):
        request_body = request_body.decode("utf-8")
    content_type = (request_content_type or "").split(";")[0].strip()

    if content_type == "text/csv":
        # One record per line, comma-separated features, no header and no label
        # column — the format preprocessing.py wrote for test_x.csv.
        rows = [r for r in request_body.strip().split("\n") if r]
        return np.array([[float(v) for v in r.split(",")] for r in rows])

    if content_type == "application/json":
        payload = json.loads(request_body)
        if isinstance(payload, dict):
            payload = payload.get("instances", payload.get("inputs", payload))
        array = np.array(payload, dtype=float)
        return array.reshape(1, -1) if array.ndim == 1 else array

    raise ValueError(f"Unsupported content type: {content_type}")


def predict_fn(input_data, model):
    import xgboost as xgb

    return model.predict(xgb.DMatrix(input_data))


def output_fn(prediction, accept):
    accept = (accept or "text/csv").split(";")[0].strip()
    values = [float(p) for p in np.atleast_1d(prediction)]
    if accept == "application/json":
        return json.dumps(values), "application/json"
    return "\n".join(str(v) for v in values), "text/csv"
"""

# log_model_artifacts preserves the local directory layout, so code/inference.py
# lands at <artifact_location>/code/inference.py.
with tempfile.TemporaryDirectory() as tmp:
    os.makedirs(os.path.join(tmp, "code"))
    with open(os.path.join(tmp, "code", "inference.py"), "w") as f:
        f.write(INFERENCE_SCRIPT)
    mlflow_client.log_model_artifacts(logged_model.model_id, tmp)

print(f"Logged code/inference.py to {logged_model.artifact_location}/code/")

#### Log the inference specification

`sagemaker_mlflow.log_inference_specification()` attaches the container image, the model data source, and the environment variables to the logged model. The instance types you list constrain where the Model Package can be deployed, so include the type you intend to use.

In [ ]:
inference_spec = {
    "Containers": [{
        "Image": xgb_training_image,
        # Serve straight from the MLflow artifact store — no repacking, no copies.
        "ModelDataSource": {
            "S3DataSource": {
                "S3Uri": logged_model.artifact_location.rstrip("/") + "/",
                "S3DataType": "S3Prefix",
                "CompressionType": "None",
            }
        },
        # Point the BYOC serving stack at the script logged above.
        "Environment": {
            "SAGEMAKER_PROGRAM": "inference.py",
            "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
        },
    }],
    "SupportedContentTypes": ["text/csv", "application/json"],
    "SupportedResponseMIMETypes": ["text/csv", "application/json"],
    "SupportedRealtimeInferenceInstanceTypes": ["ml.m5.large", "ml.m5.xlarge", "ml.r5.xlarge"],
    "SupportedTransformInstanceTypes": ["ml.m5.large", "ml.m5.xlarge"],
}

sagemaker_mlflow.log_inference_specification(
    logged_model.model_id, inference_specification=inference_spec
)
print("Inference specification logged on the MLflow model.")

In [ ]:
logged_model_artifacts = f"{presigned_url.split('/auth')[0]}/#/experiments/{experiment_id}/models/{logged_model.model_id}/artifacts?workspace=default"

In [ ]:
display(Javascript('window.open("{}");'.format(logged_model_artifacts)))

![Inference Specification for SageMaker logged as part of the model artifacts in MLflow](./img/inference-spec-mlflow.png)

#### Register, and watch the auto-sync

Register the logged model. This is the only registration in the notebook, so it becomes version 1 of `registered_model_name`. Two details make the artifact review-ready from the start:

- **Group tags.** `mlflow.register_model()` creates the registered model without tags. Creating it first with `MlflowClient.create_registered_model(tags=...)` lets the sync create the SageMaker **Model Package Group** with group-level facts (project, use case, owning team).
- **The sync tag.** The sync to the SageMaker Model Registry runs asynchronously and records the resulting Model Package ARN as a `sagemaker.model_package_arn` tag on the MLflow model version, a simpler way to find the package than searching the registry.

In [ ]:
from mlflow.exceptions import MlflowException

# Facts about the model as a whole: they become tags of the Model Package Group.
group_tags = {
    "project": "from-idea-to-prod",
    "use-case": "bank-marketing",
    "team": "data-science",
    "model-type": "xgboost-binary-classifier",
}
try:
    mlflow_client.create_registered_model(
        registered_model_name,
        tags=group_tags,
        description="Bank-marketing propensity model (XGBoost), registered from notebook 01.",
    )
    print(f"Created registered model '{registered_model_name}' with tags {sorted(group_tags)}")
except MlflowException as e:
    if e.error_code != "RESOURCE_ALREADY_EXISTS":
        raise
    print(f"Registered model '{registered_model_name}' already exists — reusing it")

registered_model_version = mlflow.register_model(
    f"models:/{logged_model.model_id}", registered_model_name
)
print(f"Registered in MLflow: {registered_model_version.name} v{registered_model_version.version}")

model_package_arn = None
for _ in range(20):
    mv = mlflow_client.get_model_version(
        registered_model_version.name, registered_model_version.version
    )
    model_package_arn = mv.tags.get("sagemaker.model_package_arn")
    if model_package_arn:
        break
    print(".", end="", flush=True)
    time.sleep(3)
if not model_package_arn:
    raise TimeoutError("SageMaker Model Package was not auto-created within the timeout")
print(f"\nSynced SageMaker Model Package: {model_package_arn}")

#### Inspect the auto-created Model Package

This is the object notebooks 03 and 04 build on. It already carries an `InferenceSpecification`, the payoff of logging the spec before registering.

<div class="alert alert-warning">
<b>SDK gap worth knowing about:</b> the typed <code>ModelPackage</code> resource deserializes the nested containers into a <code>ContainersItem</code> shape that models only 6 of the API's 16 container fields. <code>Environment</code>, <code>ImageDigest</code>, <code>ContainerHostname</code>, <code>Framework</code> and others are dropped on read, and touching <code>container.environment</code> raises <code>AttributeError</code> even though the values <i>are</i> stored in the registry. So the cell below reads the modeled fields from the typed resource and falls back to the raw <code>DescribeModelPackage</code> response for the rest.
</div>

In [ ]:
model_package = ModelPackage.get(model_package_name=model_package_arn)

# DescribeModelPackage does not return ModelPackageName for versioned packages,
# which breaks refresh()/update(). The API accepts an ARN in that field, so
# backfill it here.
model_package.model_package_name = model_package.model_package_arn

# The typed ContainersItem shape omits Environment, ImageDigest and friends, so
# read the raw response for those. One extra API call, no guessing.
described = boto3.client("sagemaker").describe_model_package(
    ModelPackageName=model_package_arn
)
raw_container = described["InferenceSpecification"]["Containers"][0]

print(f"Model package group:  {model_package.model_package_group_name}")
print(f"Version:              {model_package.model_package_version}")
print(f"Status:               {model_package.model_package_status}")
print(f"Approval status:      {described.get('ModelApprovalStatus', '(not set)')}")
print(f"Has inference spec:   {model_package.inference_specification is not None}")

print("\nInference specification, as stored in the registry")
print(f"  image:              {raw_container['Image']}")
print(f"  image digest:       {raw_container.get('ImageDigest', '(none)')}")
print(f"  model data source:  {raw_container['ModelDataSource']['S3DataSource']['S3Uri']}")
print(f"  compression:        {raw_container['ModelDataSource']['S3DataSource']['CompressionType']}")
print(f"  environment:        {raw_container.get('Environment', {})}")
print(f"  content types:      {described['InferenceSpecification']['SupportedContentTypes']}")
print(f"  instance types:     {described['InferenceSpecification']['SupportedRealtimeInferenceInstanceTypes']}")

# Fail loudly if the environment we logged did not survive the sync: without
# SAGEMAKER_PROGRAM a framework container has no inference code to run.
expected_env = inference_spec["Containers"][0]["Environment"]
synced_env = raw_container.get("Environment", {})
mismatched = {k: (v, synced_env.get(k)) for k, v in expected_env.items() if synced_env.get(k) != v}
if mismatched:
    raise RuntimeError(
        f"Environment did not round-trip through the MLflow sync: {mismatched}. "
        "The endpoint would fall back to the container's defaults."
    )
print("\nEnvironment round-tripped through the sync intact.")

The sync also generated a **draft Model Card** on the package, pre-filled with the training metrics and hyperparameters of the winning run. It comes with the auto-registration; you did not ask for it.

In [ ]:
import json

card = json.loads(described["ModelCard"]["ModelCardContent"])
training = card["training_details"]["training_job_details"]

print(f"Model card status: {described['ModelCard']['ModelCardStatus']}")
print("\nTraining metrics recorded on the card:")
for m in training["training_metrics"]:
    print(f"  {m['name']:<22}{m['value']:.4f}")
print("\nHyperparameters recorded on the card:")
for h in training["hyper_parameters"]:
    if h["value"] != "None":
        print(f"  {h['name']:<22}{h['value']}")

#### Move the version to staging: the data scientist's step

A freshly synced version has no lifecycle stage and no approval status. The data scientist sets the stage without leaving MLflow: setting the alias **`sagemakerlifecycle-<stage>-<status>`** on the model version makes the sync update the Model Package's `ModelLifeCycle`. A version holds one lifecycle alias at a time, so `set_lifecycle()` removes any existing one first.

Where provenance goes is dictated by the API: SageMaker supports tag operations on Model Package **Groups** only (`AddTags` against a version ARN fails with `ValidationException`), so:

- **Group resource tags** carry model-wide facts (project, use case, team) — what IAM `aws:ResourceTag` conditions, Resource Groups, and cost allocation see.
- **`CustomerMetadataProperties` on the version** carry per-version facts — shown on the version in Studio, and read by the notebook 02 card automation as `EvaluationMetrics`.

In [ ]:
sm_client = boto3.client("sagemaker")

# Lifecycle stage, driven from MLflow by alias. The sync applies it to the
# Model Package with the MLflow app's service role, then we wait for it to land.
set_lifecycle(mlflow_client, registered_model_version.name, registered_model_version.version,
              stage="staging", status="pending")
lifecycle = wait_for_lifecycle(sm_client, model_package_arn, "staging", "pending")

# Provenance of THIS version -> CustomerMetadataProperties on the version.
source_run = mlflow_client.get_run(best_run_id)
version_metadata = set_version_metadata(sm_client, model_package_arn, {
    "RegistrationSource": "notebook-01",
    "RegisteredBy": user_profile_name or "unknown",
    "MlflowExperiment": experiment_name,
    "TrainingJob": source_run.data.tags.get("sagemaker.job_name", "n/a"),
    "ValidationAuc": f"{metric_of(logged_model, 'validation_auc'):.4f}",
})
print("\nRecorded on the version (CustomerMetadataProperties):")
for key in sorted(version_metadata):
    print(f"  {key:<24}{version_metadata[key][:70]}")

# Facts about the model as a whole -> resource tags on the Model Package GROUP.
group_resource_tags = tag_model_package_group(sm_client, model_package_arn, group_tags)
print("\nResource tags on the Model Package Group:")
for key in sorted(group_resource_tags):
    print(f"  {key:<24}{group_resource_tags[key]}")

print_lifecycle(sm_client, model_package_arn, "\nRegistry state after the data scientist is done:")

## Pass the approval gate

The Model Package exists, sits in `staging/pending`, and is not approved. That ordering is the point of a registry: you get a reviewable artifact first, and promotion and deployment are separate, deliberate steps taken by the right people.

The rest of the notebook is the gate: first as the *data scientist*, who tries to promote to production and is denied by IAM; then as the *governance officer*, who reviews the synced evidence, promotes to `production/active`, and approves it.

Deployment is deliberately not part of this notebook. [Notebook 04](04-inference-and-monitoring.ipynb) resolves the approved version from the registry, validates its logged `inference.py` in the kernel, deploys it to a real-time endpoint with data capture, and monitors it.

### Two attributes, two enforcement points

Lifecycle stage and approval status are separate governance attributes:

- **Lifecycle stage** (`ModelLifeCycle`: `staging` or `production`, status `pending` or `active`) says *where the model is in its life*.
- **Approval status** (`ModelApprovalStatus`: `PendingManualApproval`, `Approved`, `Rejected`) says *whether it may be deployed*.

Promotion to production and approval for deployment are the governance officer's calls, and IAM keeps the data scientist out of them. Because the two attributes are enforced at different points, the cells below check them differently:

| Attempt | Route | How it is checked here |
|---|---|---|
| Promote to `production/active` | MLflow alias → sync → `UpdateModelPackage` by the **MLflow app service role** | `iam:SimulateCustomPolicy` on the data-scientist policy: IAM returns `explicitDeny` |
| Set `ModelApprovalStatus=Approved` | `UpdateModelPackage` by the **calling role** | assume the data-scientist role and call: `AccessDeniedException` |

Simulation is IAM's own evaluation engine answering the same question a live call would, and it is how the [blog's reference notebook](https://github.com/aws-samples/genai-ml-platform-examples/blob/main/operations/sagemaker-mlflow-model-registry-goverance/01_single_account_governance.ipynb) demonstrates the lifecycle guardrail. Attach this policy to the data-scientist roles *and* to the MLflow app service role to close the alias route too.

#### Data scientist: denied

Both checks below **expect** a denial. If either comes back allowed, the guardrail is not in place and the cell raises so you notice.

In [ ]:
iam_client = boto3.client("iam")
ds_policy = load_persona_policy(
    DATA_SCIENTIST,
    region=region,
    account_id=boto3.client("sts").get_caller_identity()["Account"],
    model_package_group_prefix="from-idea-to-prod-model-",
)
print("Data-scientist guardrail:")
describe_guardrail({"Statement": deny_statements(ds_policy)})

In [ ]:
# 1. Lifecycle promotion to production — the alias route runs as the MLflow app's
#    service role, so ask IAM to decide the request instead of calling the API.
for stage, status in (("staging", "pending"), ("production", "active")):
    decision, source = simulate_lifecycle_decision(iam_client, ds_policy, stage, status)
    verdict = "DENIED" if decision == "explicitDeny" else "NOT DENIED"
    print(f"\n{DATA_SCIENTIST} -> lifecycle {stage}/{status}: {verdict} "
          f"(IAM decision '{decision}', evaluated by {source})")
    if stage == "production" and decision != "explicitDeny":
        raise RuntimeError(
            "The data-scientist policy did not deny promotion to production. Check the "
            "DenyProductionPromotion statement in governance/data_scientist_policy.json."
        )

In [ ]:
# 2. Approval status IS evaluated against the caller, so this one is denied live.
ds_sm = assume_persona(data_scientist_role_arn, DATA_SCIENTIST, region=region)
allowed, message = try_update_model_package(
    ds_sm, ModelPackageArn=model_package_arn, ModelApprovalStatus="Approved"
)
print(f"\n{DATA_SCIENTIST} -> ModelApprovalStatus=Approved: {'ALLOWED' if allowed else 'DENIED'}")
if allowed:
    raise RuntimeError(
        "The data-scientist role was allowed to approve the model. Check the "
        "DenyDeploymentApproval statement in governance/data_scientist_policy.json."
    )
print(f"  {message}")

print_lifecycle(sm_client, model_package_arn, "\nUnchanged, as it should be:")

#### Governance officer: review, promote, approve

The officer does not need MLflow open: the synced version already carries the training metrics, evaluation card, lineage, and the provenance the data scientist recorded. The cell prints what a reviewer looks at first, then makes two distinct decisions in order:

- `ModelLifeCycle = production/active`: the version is now a production-stage candidate in this group. Set through the lifecycle alias, same mechanism as staging.
- `ModelApprovalStatus = Approved`: the version may be deployed. This turns on the **Deploy** button in Studio and is what the notebook 02 card automation reacts to (card status `PendingReview` → `Approved`). Made as the governance-officer role, whose policy carries no deny.

The decision is also recorded on the version as `ApprovedBy` and `ApprovedAt`, so the audit trail sits next to the model, not only in CloudTrail.

In [ ]:
go_sm = assume_persona(governance_officer_role_arn, GOVERNANCE_OFFICER, region=region)

# --- Review: what the officer sees in the registry, without opening MLflow.
described = go_sm.describe_model_package(ModelPackageName=model_package_arn)
card = json.loads(described["ModelCard"]["ModelCardContent"])
training = card.get("training_details", {}).get("training_job_details", {})
card_metrics = {m["name"]: m["value"] for m in training.get("training_metrics", [])}
provenance = described.get("CustomerMetadataProperties", {})
print("Evidence on the Model Package version:")
print(f"  validation_auc:   {card_metrics.get('validation_auc', float('nan')):.4f}")
print(f"  card status:      {described['ModelCard']['ModelCardStatus']}")
print(f"  registered by:    {provenance.get('RegisteredBy')} from {provenance.get('RegistrationSource')}")
print(f"  training job:     {provenance.get('TrainingJob')}")
print(f"  inference spec:   {'present' if described.get('InferenceSpecification') else 'MISSING'}")
print(f"  group tags:       {model_package_group_tags(go_sm, model_package_arn)}")

# --- Promote: lifecycle to production/active, through the alias.
set_lifecycle(mlflow_client, registered_model_version.name, registered_model_version.version,
              stage="production", status="active")
wait_for_lifecycle(sm_client, model_package_arn, "production", "active")

In [ ]:
# --- Approve for deployment: a direct call, made as the governance officer.
summary = approve_for_deployment(
    go_sm,
    model_package_arn,
    approved_by=f"{GOVERNANCE_OFFICER}:{user_profile_name}",
    note=f"Validation AUC {card_metrics.get('validation_auc', 0):.4f} meets the 0.75 bar; inference spec and card reviewed.",
)
print_lifecycle(go_sm, model_package_arn, "\nAfter the governance officer's decision:")
assert summary["approval_status"] == "Approved" and summary["stage"] == "production"

In practice the governance officer usually does this in the **Studio Models UI**: open the version, check the *Train*, *Evaluate*, and *Lineage* tabs, then update the stage and approval status from the *Details* tab. Same API calls, same IAM evaluation.

In [ ]:
display(
    HTML('<b>See <a target="top" href="https://studio-{}.studio.{}.sagemaker.aws/registry/registered-models/{}/versions/version-{}">the promoted Model Package version</a> in the Studio UI</b>'.format(
            domain_id, region, model_package.model_package_group_name, model_package.model_package_version))
)

Back as yourself, confirm the registry reflects the officer's decision; the typed `ModelPackage` resource fetched earlier only needs a `refresh()`.

Optionally the officer can **freeze** the model: `freeze_model_package_group()` tags the group `frozen=true`, and [`governance/freeze_policy.json`](governance/freeze_policy.json) turns that tag into a deny on further `UpdateModelPackage` / `CreateModelPackage`. This workshop leaves the group unfrozen, because notebook 03 registers another version into it.

In [ ]:
model_package.refresh()
print(f"Approval status:  {model_package.model_approval_status}")
print(f"Lifecycle:        {model_package.model_life_cycle}")

# The data scientist still cannot undo the approval.
allowed, message = try_update_model_package(
    ds_sm, ModelPackageArn=model_package_arn, ModelApprovalStatus="Approved"
)
print(f"\n{DATA_SCIENTIST} -> re-approve: {'ALLOWED' if allowed else 'DENIED'}")

# To freeze the group instead of leaving it open, uncomment:
# freeze_model_package_group(go_sm, model_package_arn)

## Explore the full lineage of the model version

In [ ]:
display(
    HTML('<b>See <a target="top" href="https://studio-{}.studio.{}.sagemaker.aws/registry/registered-models/{}/versions/version-{}/lineage">the model package version lineage</a> in the Studio UI</b>'.format(
            domain_id, region, model_package.model_package_group_name, model_package.model_package_version))
)

![](img/model-version-lineage.png)

## Summary

In this notebook you:

**Part 1 — in the notebook**
- Explored the dataset, engineered features, and trained XGBoost locally with cross-validation and a `max_depth` sweep
- Tracked runs, parameters, metrics, and artifacts in managed MLflow, and registered nothing, because a kernel-trained prototype is not a deployment candidate

**Part 2 — on SageMaker AI managed infrastructure**
- **Processing**: ran the feature engineering once as a `FrameworkProcessor` job, producing the canonical train/validation/test/baseline datasets in S3
- **Training**: turned a plain function into a job with `@remote`, smoke-tested `train.py` in `LOCAL_CONTAINER` mode, and ran it at scale with `ModelTrainer`
- **Explainability**: selected the best candidate with `mlflow.search_logged_models`, then ran [`pipeline_steps/explain.py`](pipeline_steps/explain.py) as a managed `@remote` job
- **Registration**: logged an `inference.py` and inference specification, so the auto-synced Model Package arrived deployable
- **Approval gate**: confirmed the data-scientist denials with `iam:SimulateCustomPolicy` and live, then promoted and approved as the governance officer

Notebook 03 combines these steps into an **automated ML pipeline**; notebook 04 deploys the approved package from the registry.

## Continue with notebook 02

Open [notebook 02](02-model-card-automation.ipynb). It picks up the Model Package you just registered and automates completion of its Model Card, for this version and every version the notebook 03 pipeline registers, keeping the card status in step with the approval workflow.

## Further development ideas for your real-world projects
- Learn more about [managing experiments with MLflow](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html)
- Try no-code [SageMaker Canvas](https://docs.aws.amazon.com/sagemaker/latest/dg/canvas.html) for analysis, automated ML, and predictions
- Improve training with [automated hyperparameter tuning](https://docs.aws.amazon.com/sagemaker/latest/dg/automatic-model-tuning.html)
- Attach the `sagemaker:ModelLifeCycle/stage` guardrail to the **MLflow app service role** to close the alias route, apply [`governance/freeze_policy.json`](governance/freeze_policy.json) to lock approved models, and give each persona its own SageMaker user profile (or account, per Part 2 of the [governance blog series](https://aws.amazon.com/blogs/machine-learning/govern-models-with-mlflow-and-amazon-sagemaker-ai-model-registry-sync-part-1/)) instead of assuming roles from one notebook

## Additional resources
- [Managing ML experiments with MLflow in SageMaker](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html)
- [MLflow Apps in SageMaker](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow-app-setup.html)
- [Using Docker containers with SageMaker](https://docs.aws.amazon.com/sagemaker/latest/dg/docker-containers.html)
- [SageMaker JupyterLab guide](https://docs.aws.amazon.com/sagemaker/latest/dg/studio-updated-jl.html)
- [Operationalize your Amazon SageMaker Studio notebooks as scheduled notebook jobs](https://aws.amazon.com/blogs/machine-learning/operationalize-your-amazon-sagemaker-studio-notebooks-as-scheduled-notebook-jobs/)
- [scikit-learn dataset transformations](https://scikit-learn.org/stable/data_transforms.html)
- [Amazon SageMaker Immersion Day](https://catalog.us-east-1.prod.workshops.aws/workshops/63069e26-921c-4ce1-9cc7-dd882ff62575/en-US)

### Hands-on examples
- [Official Amazon SageMaker Examples repository](https://github.com/aws/amazon-sagemaker-examples)
- [Community Amazon SageMaker Examples repository](https://github.com/aws/amazon-sagemaker-examples-community)

# Shutdown kernel

In [ ]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>